<a href="https://colab.research.google.com/github/MiriamSchirmer/online-harm-nlp-ws2627/blob/main/notebooks/01_Tokenization_and_Regex_Mini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini trial: from strings to tokens and regex

**Applied NLP for Online Harm Detection**, University of Konstanz, WS 2026/27

In this notebook you first learn how a notebook works and what a **string** is. Then we run the examples from today's slides: tricky "words", the pre-processing pipeline, and regular expressions, and apply them to a text from the reading (Fortuna & Nunes 2018).

✏️ marks the cells where you experiment.

## 0. What is a cell?

A notebook consists of **cells**:

- **Text cells** (like this one) explain things. You only read them.
- **Code cells** (grey boxes) contain Python code. Click into one and press `Shift + Enter` (or the ▶ button on the left) to run it. The result appears directly below the cell.

Run the cells **from top to bottom**: later cells often use something an earlier cell created. If something goes wrong, use *Runtime → Run all* to start again from the beginning.

Try it: run the cell below.

In [1]:
print("Hello, Konstanz!")
2 + 3

Hello, Konstanz!


5

✏️ Change the text between the quotes or the numbers and run the cell again.

## 1. What is a string?

For Python, text is a **string**: a sequence of characters between quotes. With `=` we give the string a name (a **variable**) so we can use it again later.

In [2]:
sentence = "I preferred LOS ANGELES."

print(sentence)
print(len(sentence))     # number of characters, spaces and punctuation included
print(sentence[0])       # the first character (Python starts counting at 0)
print(sentence[-1])      # the last character

I preferred LOS ANGELES.
24
I
.


Strings come with useful built-in tools (**methods**), which we write after a dot:

In [3]:
print(sentence.lower())           # everything in lower case
print(sentence.upper())           # everything in upper case
print(sentence.replace("LOS ANGELES", "Konstanz"))
print(sentence.split())           # cut at the spaces: a list of pieces
print("ANGELES" in sentence)      # does the string contain this? True / False

i preferred los angeles.
I PREFERRED LOS ANGELES.
I preferred Konstanz.
['I', 'preferred', 'LOS', 'ANGELES.']
True


### ✏️ Your turn

Write your own sentence between the quotes and run the cell. How many characters does it have? What does `.split()` do with punctuation?

In [ ]:
my_sentence = "Write your own sentence here!"

print(len(my_sentence))
print(my_sentence.split())

## 2. Where defining words gets weird

On the slides we asked: what counts as one word? The easiest tokenizer cuts at the spaces with `.split()`. Let's try it on the examples from the slides.

In [4]:
examples = [
    "I do uh main- mainly business data processing",
    "I'm from the U.S.A.",
    "I've been in New York, but I threw up.",
]

for e in examples:
    print(e.split())

['I', 'do', 'uh', 'main-', 'mainly', 'business', 'data', 'processing']
["I'm", 'from', 'the', 'U.S.A.']
["I've", 'been', 'in', 'New', 'York,', 'but', 'I', 'threw', 'up.']


Look at the output:

- `uh` and `main-` become tokens, although they are a filled pause and a fragment.
- `I'm` stays one token: is it one word or two (*I* + *am*)?
- `York,` and `up.` keep their punctuation, so `York,` and `York` would be counted as different words.
- `New` and `York` are separated, although we need both to understand the concept.

How could each of these affect a text analysis?

## 3. The pre-processing pipeline from the slides

We take the example sentence from the slides and apply one step after another. Most steps use the string methods from section 1. Step 3 uses `re.sub(pattern, replacement, text)`, which replaces everything that matches a **pattern**: `\d` means "a digit" (more on patterns in section 4).

In [5]:
import re

text = "<div id=«text»> I’ve been in New York in 2011, but didn’t like it. I preferred LOS ANGELES.</div>"

# 1. Formatting removal: delete the HTML tags
step1 = text.replace("<div id=«text»>", "").replace("</div>", "").strip()
print("1. formatting removed:", step1)

# 2. Lower case
step2 = step1.lower()
print("2. lower case:        ", step2)

# 3. Numbers normalization: every digit becomes 0
step3 = re.sub(r"\d", "0", step2)
print("3. numbers:           ", step3)

# 4. Join collocations: multi-word expressions become one token
step4 = step3.replace("new york", "new_york").replace("los angeles", "los_angeles")
print("4. collocations:      ", step4)

1. formatting removed: I’ve been in New York in 2011, but didn’t like it. I preferred LOS ANGELES.
2. lower case:         i’ve been in new york in 2011, but didn’t like it. i preferred los angeles.
3. numbers:            i’ve been in new york in 0000, but didn’t like it. i preferred los angeles.
4. collocations:       i’ve been in new_york in 0000, but didn’t like it. i preferred los_angeles.


## 4. Regular expressions: the examples from the slides

A **regular expression** (regex) is a pattern that describes text. `re.findall(pattern, text)` returns every piece of the text that matches. Always write patterns as raw strings: `r"..."`.

In [6]:
print(re.findall(r"Buttercup", "I’m called little Buttercup"))     # exact characters
print(re.findall(r"[mM]ary", "Mary and mary"))                     # [ ]: one of these characters
print(re.findall(r"[A-Z]", "Drenched Blossoms"))                   # a range: one capital letter
print(re.findall(r"[0-9]", "Chapter 1: Down the Rabbit Hole"))     # one digit
print(re.findall(r"cats?", "one cat, two cats"))                   # ?: the s is optional
print(re.findall(r"o+", "so sooooo"))                              # +: one or more

['Buttercup']
['Mary', 'mary']
['D', 'B']
['1']
['cat', 'cats']
['o', 'ooooo']


**Word boundaries** (`\b`) make sure we match whole words only:

In [7]:
s = "My cat is a catastrophe. He acts like a wildcat."

print(re.findall(r"cat", s))          # cat in cat, catastrophe and wildcat
print(re.findall(r"cat\b", s))        # cat and wildcat: the word must END after cat
print(re.findall(r"\bcat\b", s))      # only the full word cat

['cat', 'cat', 'cat']
['cat', 'cat']
['cat']


### ✏️ Your turn

See what happens if you add/remove digits or words, change capitalization, etc. in the examples above.

#### 📋 Regex cheat sheet

| Pattern | Matches | Example |
|---|---|---|
| `abc` | exactly these characters | `cat` finds *cat* in *concatenate* |
| `[abc]` | one of the characters in brackets | `[tT]he` → *the*, *The* |
| `[a-z]`, `[A-Z]`, `[0-9]` | one lowercase letter, capital letter, digit | |
| `\d` | a digit | `\d` → *2*, *0*, *1*, *1* |
| `\w` | a letter, digit or `_` | |
| `.` | any one character | `c.t` → *cat*, *cut* |
| `+` | the previous thing **one or more** times | `\w+` → a whole word, `so+` → *sooooo* |
| `?` | the previous thing is **optional** | `cats?` → *cat*, *cats* |
| `\b` | word boundary (start or end of a word) | `\bcat\b` → *cat* but not *wildcat* |
| `a\|b` | `a` **or** `b` | `cat\|dog` |

`re.findall(pattern, text)` returns all matches, `re.sub(pattern, new, text)` replaces them. Test patterns interactively at [regex101.com](https://regex101.com) (choose *Python*).

## 5. The definition of hate speech from the slides

Now a real text: the definition of hate speech from the slide *Defining online harm*, taken from today's reading Fortuna & Nunes (2018), *A Survey on Automatic Detection of Hate Speech in Text*, ACM Computing Surveys 51 (4).


In [10]:
# Definition of hate speech from Fortuna & Nunes (2018)
text = """Hate speech is language that attacks or diminishes, that incites violence or hate against groups,
based on specific characteristics such as physical appearance, religion, descent, national or ethnic origin,
sexual orientation, gender identity or other, and it can occur with different linguistic styles,
even in subtle forms or when humour is used."""

✏️ Find all words that end in *-ion* (e.g. *religion*). Hint: `\w+` followed by `ion`, with word boundaries.

In [11]:
print(re.findall(r"YOUR PATTERN HERE", text))

[]


In [13]:
#@title Solution (double-click to show)
print(re.findall(r"\b\w+ion\b", text))

['religion', 'orientation']


## Takeaways

- For Python, text is a **string**: a sequence of characters. Everything we do in NLP starts from there.
- How we tokenize and pre-process decides what a computer can count and compare, and every step can also remove information we need.
- Regex is great for finding and cleaning patterns, but a word list cannot understand context.
- Next: learning to detect harmful language from labelled examples (**supervised classification**).